# Разведочный анализ PM2.5 (EDA)

**Цель.** Увидеть закономерности, которые потом проверим тестами (H1 — сезон, H2 — погода, H3 — суточный цикл), и оценить масштаб превышений нормы ВОЗ.

**Входы:** `data/processed/hourly.csv` (часы) и `data/processed/daily.csv` (сутки).
**Выходы:** графики в `figures/` (`00_pm25_distribution.png` и далее).

EDA — это разведка: мы смотрим и описываем, но не доказываем. Выводы подтвердим тестами в задаче 5.1, чтобы не «ловить звёздочки» на графиках.

**Зерно данных.** Суточные графики и статистика — по таблице `daily` (только сутки со средним `pm25_mean`); суточный цикл — по часовой таблице `hourly` (только часы со значением).

In [1]:
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sys.path.append("..")
from src import clean, eda, features

HOURLY_PATH = "../data/processed/hourly.csv"
DAILY_PATH = "../data/processed/daily.csv"
FIGURES_DIR = "../figures"

WHO_LIMIT = features.WHO_DAILY_LIMIT
SEASON_ORDER = ["winter", "spring", "summer", "autumn"]
HEATING_LABELS = {True: "Отопительный сезон", False: "Вне сезона"}

# Единая тема графиков: палитра, удобная для дальтоников
sns.set_theme(style="whitegrid", context="talk", palette="colorblind")

## Шаг 1. Читаем таблицы и проверяем контракты

Читаем готовые таблицы функциями проекта и сразу проверяем их `check_*`: если контракт нарушен, дальше смотреть нечего. Суточные графики строим только по суткам, где есть среднее PM2.5, поэтому выделяем их в `days`.

In [2]:
hourly = clean.read_clean_csv(HOURLY_PATH)
features.check_hourly(hourly)
daily = features.read_daily_csv(DAILY_PATH)
features.check_daily(daily)

print("hourly:", hourly.shape)
print("daily:", daily.shape)

# Сутки со средним — с ними работают суточные графики
days = daily[daily["pm25_mean"].notna()]
print("суток со средним PM2.5:", len(days))

hourly: (49075, 18)
daily: (2044, 16)
суток со средним PM2.5: 1534


## Шаг 2. Сколько у нас данных

Средние и медианы pandas считает по непустым значениям. Мы не заполняем пропуски заново: короткие дыры уже заполнены на этапе очистки, длинные остаются честно пустыми. Поэтому важно знать, сколько значений стоит за каждым числом: сколько часов и суток пустые и как сутки со средним распределены по годам и месяцам.

In [3]:
hours_missing = hourly["pm25"].isna().sum()
print("часов без pm25:", hours_missing, f"({hours_missing / len(hourly):.2%})")
print("часов, заполненных интерполяцией:", hourly["pm25_interpolated"].sum())
print("суток со средним:", len(days), "| без среднего:", len(daily) - len(days))

# Сколько суток со средним в каждом году и месяце
days_by_year_month = days.assign(year=days["date"].dt.year).pivot_table(
    values="pm25_mean", index="year", columns="month", aggfunc="count"
)
print(days_by_year_month)

часов без pm25: 10235 (20.86%)
часов, заполненных интерполяцией: 1148
суток со средним: 1534 | без среднего: 510
month    1     2     3     4     5     6     7     8     9     10    11    12
year                                                                         
2020    NaN   NaN   NaN  17.0  20.0  17.0  18.0  22.0  24.0  17.0  23.0  23.0
2021   21.0  18.0  22.0  18.0  12.0   9.0   6.0  17.0  14.0   9.0  23.0  27.0
2022   20.0  26.0  26.0  26.0   NaN   NaN   NaN   NaN  22.0  28.0  28.0  20.0
2023   24.0   2.0   9.0  26.0  31.0  30.0  29.0  31.0  25.0  31.0  30.0  31.0
2024   26.0  28.0  31.0  30.0  31.0  30.0  31.0  31.0  26.0  22.0  30.0  31.0
2025   31.0  28.0  31.0  30.0  31.0  30.0  31.0  31.0  28.0  31.0  13.0   NaN


**Что видим.** (вывод шага 2) Без `pm25` 10 235 часов из 49 075 (20.86 %), ещё 1 148 часов заполнены интерполяцией на этапе очистки. Суток со средним 1 534, без среднего 510. Главная дыра — май–август 2022: в таблице «год × месяц» там пусто, а в феврале–марте 2023 и ноябре 2025 мало суток (2, 9 и 13). Поэтому месячные статистики объединяют разное число лет, и это надо помнить при выводах.

## Шаг 3. Описательная статистика PM2.5

Считаем основные числа: среднее, медиану, квартили. PM2.5 скошен вправо (редкие пики смога), поэтому главные числа — медиана и межквартильный размах; среднее тянется за пиками. Сравниваем сезоны, отопительный сезон и месяцы.

In [4]:
print("Сутки, pm25_mean:")
print(days["pm25_mean"].describe())
print()
print("Часы, pm25:")
print(hourly["pm25"].describe())

Сутки, pm25_mean:
count    1534.000000
mean       29.065487
std        30.420802
min         3.289474
25%        11.010417
50%        16.871711
75%        32.526042
max       229.000000
Name: pm25_mean, dtype: float64

Часы, pm25:
count    38840.000000
mean        29.480947
std         37.746807
min          0.000000
25%          9.000000
50%         15.000000
75%         31.000000
max        421.000000
Name: pm25, dtype: float64


In [5]:
print("По сезонам (суточное среднее):")
print(days.groupby("season")["pm25_mean"].describe().reindex(SEASON_ORDER))
print()
print("По отопительному сезону:")
print(days.groupby("heating_season")["pm25_mean"].describe())

По сезонам (суточное среднее):
        count       mean        std        min        25%        50%  \
season                                                                 
winter  356.0  65.285704  40.476166  10.020833  32.757292  57.291667   
spring  391.0  17.597151  10.317504   4.021739  10.770833  14.375000   
summer  363.0  10.775799   3.751422   3.289474   8.458333  10.333333   
autumn  424.0  24.888314  18.397583   4.916667  13.875000  19.302083   

              75%         max  
season                         
winter  83.156250  229.000000  
spring  21.572917   79.416667  
summer  12.375000   33.625000  
autumn  29.489583  122.000000  

По отопительному сезону:
                count       mean        std       min        25%        50%  \
heating_season                                                                
False           763.0  12.617389   5.241042  3.289474   9.048611  11.375000   
True            771.0  45.342918  35.803281  4.500000  19.750000  32.208333   

 

In [6]:
print("По месяцам:")
print(days.groupby("month")["pm25_mean"].describe())

По месяцам:
       count       mean        std        min        25%        50%  \
month                                                                 
1      122.0  66.969343  43.870727  10.020833  32.354167  56.500000   
2      102.0  62.189044  29.424741  13.250000  46.223958  58.250000   
3      119.0  25.647255  13.094606   5.583333  16.687500  23.333333   
4      147.0  16.349315   6.595765   4.076389  11.458333  14.825000   
5      125.0  11.400908   4.203654   4.021739   8.583333  11.000000   
6      116.0  11.637772   4.418271   4.111111   8.645833  10.937500   
7      115.0  10.128128   3.414945   3.289474   7.979167   9.666667   
8      132.0  10.582567   3.246550   5.000000   8.657895  10.479167   
9      139.0  14.694215   5.239754   4.916667  10.718750  14.333333   
10     138.0  21.461171   8.767164   6.333333  15.062500  20.458333   
11     147.0  37.744951  24.619075   6.083333  20.354167  31.416667   
12     132.0  66.122487  44.538203  11.583333  29.934896  57.4583

In [7]:
# Скошенность: среднее больше медианы и положительный skew — длинный правый хвост
print("суточный pm25_mean: среднее", round(days["pm25_mean"].mean(), 1),
      "| медиана", round(days["pm25_mean"].median(), 1),
      "| skew", round(days["pm25_mean"].skew(), 2))
print("часовой pm25: среднее", round(hourly["pm25"].mean(), 1),
      "| медиана", round(hourly["pm25"].median(), 1),
      "| skew", round(hourly["pm25"].skew(), 2))

суточный pm25_mean: среднее 29.1 | медиана 16.9 | skew 2.48
часовой pm25: среднее 29.5 | медиана 15.0 | skew 2.91


**Что видим.** (вывод шага 3) Распределение скошено вправо: среднее суточного PM2.5 29.1 µg/m³ больше медианы 16.9, skew 2.48 (у часовых данных 29.5, 15.0 и 2.91). По сезонам медиана зимой 57.3 µg/m³, весной 14.4, летом 10.3, осенью 19.3. Отопительный сезон: медиана 32.2 µg/m³ (771 сутки), вне сезона 11.4 (763 суток); максимум суток 229 µg/m³. По месяцам самое высокое среднее в январе (67.0 µg/m³), самая высокая медиана в феврале (58.3), самые чистые месяцы — июль и август (медианы 9.7 и 10.5). Это описание; различие сезонов проверим тестом в 5.1.

## Шаг 4. Распределение: почему дальше Mann–Whitney

Слева — гистограмма суточного PM2.5 с линией нормы ВОЗ. Справа — ECDF (эмпирическая функция распределения): точка (x, y) значит «в доле y дней PM2.5 не превышал x»; кривая правее — значения выше. Две кривые — отопительный сезон и остальное время. ECDF сравнивает распределения целиком и не зависит от числа столбцов гистограммы. Если распределение скошено, сравнивать сезоны лучше непараметрическим тестом Mann–Whitney, а не t-тестом.

In [8]:
mean_pm25 = days["pm25_mean"].mean()
median_pm25 = days["pm25_mean"].median()
skew_pm25 = days["pm25_mean"].skew()
print(f"среднее {mean_pm25:.1f}, медиана {median_pm25:.1f}, skew {skew_pm25:.2f}")

fig, (ax_hist, ax_ecdf) = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(days["pm25_mean"], bins=50, ax=ax_hist)
ax_hist.axvline(WHO_LIMIT, color="black", linestyle="--", label="Норма ВОЗ 15 µg/m³")
ax_hist.set_xlabel("PM2.5, µg/m³")
ax_hist.set_ylabel("Число дней")
ax_hist.legend()

sns.ecdfplot(data=days, x="pm25_mean", hue=days["heating_season"].map(HEATING_LABELS), ax=ax_ecdf)
ax_ecdf.axvline(WHO_LIMIT, color="black", linestyle="--")
ax_ecdf.set_xlabel("PM2.5, µg/m³")
ax_ecdf.set_ylabel("Доля дней с PM2.5 ≤ x")
ax_ecdf.get_legend().set_title("Период")

fig.suptitle(f"PM2.5 скошен вправо: среднее {mean_pm25:.0f} больше медианы {median_pm25:.0f} µg/m³")
fig.savefig(f"{FIGURES_DIR}/00_pm25_distribution.png", dpi=150, bbox_inches="tight")
plt.close(fig)

среднее 29.1, медиана 16.9, skew 2.48


**Что видим.** (вывод шага 4) Гистограмма показывает длинный правый хвост: среднее 29.1 µg/m³ больше медианы 16.9, skew 2.48. Кривая ECDF отопительного сезона идёт правее кривой остального времени — в отопительный сезон значения выше. Раз распределение скошено, для сравнения сезонов в 5.1 берём непараметрический тест Mann–Whitney.

![График 0](../figures/00_pm25_distribution.png)

## Шаг 5. График 1 — как меняется загрязнение во времени

Вопрос: как PM2.5 меняется по датам. Для времени подходит линейный график: тонкая линия — сутки, толстая — среднее за 30 дней (сглаживает шум и показывает сезонность), пунктир — норма ВОЗ. Пропуски остаются разрывами линии: это честнее, чем рисовать выдуманные значения.

In [9]:
# Скользящее среднее: строка таблицы = календарные сутки подряд (проверено check_daily),
# поэтому окно 30 строк = 30 дней; если в окне меньше 15 дней со значением — среднее пустое
rolling_30 = daily["pm25_mean"].rolling(30, min_periods=15, center=True).mean()

peak_date = daily.loc[rolling_30.idxmax(), "date"]
peak_value = rolling_30.max()
print(f"максимум 30-дневного среднего: {peak_value:.1f} µg/m³, дата {peak_date.date()}")
print("максимум 30-дневного среднего по календарным годам:")
print(rolling_30.groupby(daily["date"].dt.year).max().round(1))

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(daily["date"], daily["pm25_mean"], linewidth=0.8, alpha=0.5, label="Сутки")
ax.plot(daily["date"], rolling_30, linewidth=2.5, label="Среднее за 30 дней")
ax.axhline(WHO_LIMIT, color="black", linestyle="--", label="Норма ВОЗ 15 µg/m³")
ax.set_xlabel("Дата")
ax.set_ylabel("PM2.5, µg/m³")
ax.legend()
ax.set_title(f"Каждый год 30-дневное среднее PM2.5 выше нормы ВОЗ: до {peak_value:.0f} µg/m³ ({peak_value / WHO_LIMIT:.0f}× нормы)")
fig.savefig(f"{FIGURES_DIR}/01_timeseries.png", dpi=150, bbox_inches="tight")
plt.close(fig)


максимум 30-дневного среднего: 135.3 µg/m³, дата 2020-12-26
максимум 30-дневного среднего по календарным годам:
date
2020    135.3
2021    132.7
2022     83.7
2023    119.1
2024     61.5
2025     74.4
Name: pm25_mean, dtype: float64


**Что видим.** (вывод шага 5) Каждую зиму 30-дневное среднее поднимается далеко выше нормы ВОЗ: максимум 135.3 µg/m³ (около 9 норм) в конце декабря 2020. Максимумы по календарным годам: 2020 — 135.3, 2021 — 132.7, 2022 — 83.7, 2023 — 119.1, 2024 — 61.5, 2025 — 74.4 µg/m³, то есть во все годы выше 15. Летом среднее опускается к норме. Разрыв линии весной–летом 2022 года — это пропуск данных, а не чистый воздух.

![График 1](../figures/01_timeseries.png)

## Шаг 6. График 2 — какие месяцы хуже всего

Вопрос: как распределено суточное PM2.5 по месяцам. Для сравнения распределений по группам подходит boxplot: ящик — межквартильный размах, линия внутри — медиана. Выбросы не прячем: пики смога — это и есть явление, которое мы изучаем.

In [10]:
import warnings

# Предупреждение внутри seaborn 0.13 на matplotlib 3.11 (параметр vert), от нашего кода не зависит;
# глушим только его
warnings.filterwarnings("ignore", message="vert: bool")

month_medians = days.groupby("month")["pm25_mean"].median()
worst_month = month_medians.idxmax()
worst_median = month_medians.max()
print(month_medians.round(1))
print(f"худший месяц: {eda.MONTH_LABELS[worst_month]}, медиана {worst_median:.1f} µg/m³")

fig, ax = plt.subplots(figsize=(16, 6))
sns.boxplot(data=days, x="month", y="pm25_mean", color=sns.color_palette()[0], ax=ax)
ax.axhline(WHO_LIMIT, color="black", linestyle="--", label="Норма ВОЗ 15 µg/m³")
ax.set_xticks(range(12))
ax.set_xticklabels([eda.MONTH_LABELS[m] for m in range(1, 13)])
ax.set_xlabel("Месяц")
ax.set_ylabel("PM2.5 за сутки, µg/m³")
ax.legend()
ax.set_title(f"Хуже всего {eda.MONTH_LABELS[worst_month]}: медиана {worst_median:.0f} µg/m³ — в {worst_median / WHO_LIMIT:.1f} раза выше нормы ВОЗ")
fig.savefig(f"{FIGURES_DIR}/02_monthly_box.png", dpi=150, bbox_inches="tight")
plt.close(fig)


month
1     56.5
2     58.2
3     23.3
4     14.8
5     11.0
6     10.9
7      9.7
8     10.5
9     14.3
10    20.5
11    31.4
12    57.5
Name: pm25_mean, dtype: float64
худший месяц: фев, медиана 58.2 µg/m³


**Что видим.** (вывод шага 6) Медиана выше нормы ВОЗ в январе (56.5), феврале (58.2), марте (23.3), октябре (20.5), ноябре (31.4) и декабре (57.5 µg/m³); хуже всего февраль — 58.2 µg/m³, в 3.9 раза выше нормы. В апреле (14.8) и сентябре (14.3) медиана около нормы, с мая по август — ниже (9.7–11.0). Выбросы есть почти во всех месяцах: это отдельные дни смога. Месяцы объединяют разное число лет (дыра лета 2022).

![График 2](../figures/02_monthly_box.png)

## Шаг 7. График 7 — масштаб проблемы: доля дней выше нормы ВОЗ

Считаем долю суток со средним выше 15 µg/m³ по месяцам; месяцы сортируем по убыванию доли. Сутки без среднего не считаем: «нет данных» не значит «не превышено» (в таблице это `<NA>`, и `count`/`sum` его пропускают).

In [11]:
overall_share = days["who_exceed"].mean()
print(f"превышений: {days['who_exceed'].sum()} из {len(days)} суток ({overall_share:.1%})")

share = eda.who_exceed_share_by_month(daily)
print(share)
print("сумма days_known:", share["days_known"].sum())

worst = share.iloc[0]
best = share.iloc[-1]
print(f"худший месяц: {eda.MONTH_LABELS[worst['month']]} {worst['share']:.1%}, "
      f"лучший: {eda.MONTH_LABELS[best['month']]} {best['share']:.1%}")

fig, ax = plt.subplots(figsize=(16, 6))
bars = ax.bar([eda.MONTH_LABELS[m] for m in share["month"]], share["share"] * 100,
              color=sns.color_palette()[0])
ax.bar_label(bars, fmt="%.0f%%")
ax.set_xlabel("Месяц")
ax.set_ylabel("Доля дней с PM2.5 > 15 µg/m³, %")
ax.set_title(f"Норма ВОЗ превышена в {overall_share:.0%} дней; в {eda.MONTH_LABELS[worst['month']]} — в {worst['share']:.0%}, в {eda.MONTH_LABELS[best['month']]} — в {best['share']:.0%}")
fig.savefig(f"{FIGURES_DIR}/07_who_exceed.png", dpi=150, bbox_inches="tight")
plt.close(fig)


превышений: 845 из 1534 суток (55.1%)
    month  days_known  days_exceed     share
0      12         132          127  0.962121
1       2         102           98  0.960784
2       1         122          117  0.959016
3      11         147          126  0.857143
4       3         119           94  0.789916
5      10         138          104  0.753623
6       4         147           72  0.489796
7       9         139           62  0.446043
8       6         116           16  0.137931
9       5         125           15      0.12
10      7         115            9  0.078261
11      8         132            5  0.037879
сумма days_known: 1534
худший месяц: дек 96.2%, лучший: авг 3.8%


**Что видим.** (вывод шага 7) Норма ВОЗ превышена в 845 сутках из 1 534 (55.1 %). Хуже всего декабрь — 96 % дней (127 из 132), далее февраль (96 %, 98 из 102) и январь (96 %, 117 из 122); лучше всего август — 4 % (5 из 132). Сумма суток со средним по месяцам равна 1 534. Сутки без среднего в долю не входят.

![График 7](../figures/07_who_exceed.png)

## Шаг 8. График 3 — когда в течение суток пик и зависит ли он от сезона

Вопрос: в какой час и в какой месяц PM2.5 выше всего. Для двух категорий (час × месяц) и одной величины подходит тепловая карта: цвет — средний PM2.5. Считаем по часовой таблице, час — местное время Алматы (признак `hour` из задачи 3.1, смена пояса 1 марта 2024 уже учтена). Таблицу строим через `pivot_table` (лекция 5). Рядом считаем, сколько часов стоит за каждой клеткой, чтобы знать, насколько ей можно верить.

In [12]:
table = eda.hour_month_table(hourly)
counts = eda.hour_month_table(hourly, aggfunc="count")
print("размер таблицы:", table.shape)
print("часов в клетке: минимум", counts.min().min(), ", максимум", counts.max().max())

peak_month, peak_hour = table.stack().idxmax()[1], table.stack().idxmax()[0]
peak_value = table.stack().max()
print(f"максимум: месяц {eda.MONTH_LABELS[peak_month]}, час {peak_hour}:00, {peak_value:.1f} µg/m³")
print("минимум таблицы:", round(table.stack().min(), 1), "µg/m³")

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(table.rename(columns=eda.MONTH_LABELS), cmap="viridis",
            cbar_kws={"label": "PM2.5, µg/m³"}, ax=ax)
ax.set_xlabel("Месяц")
ax.set_ylabel("Час (местное время Алматы)")
ax.set_title(f"Пик PM2.5 — после полудня в холодные месяцы: {eda.MONTH_LABELS[peak_month]}, {peak_hour}:00 — {peak_value:.0f} µg/m³")
fig.savefig(f"{FIGURES_DIR}/03_hour_month_heatmap.png", dpi=150, bbox_inches="tight")
plt.close(fig)


размер таблицы: (24, 12)
часов в клетке: минимум 107 , максимум 159
максимум: месяц янв, час 15:00, 95.3 µg/m³
минимум таблицы: 7.7 µg/m³


**Что видим.** (вывод шага 8) Самое высокое среднее — январь, 15:00: 95.3 µg/m³; самое низкое в таблице — 7.7 µg/m³. Зимой (декабрь, январь) суточный цикл выражен: минимум рано утром (6–7 часов), потом рост с 9–10 часов и плато около 90 µg/m³ с 12 до 18 часов, к ночи спад. Летом (июль) цикл почти плоский: 8–13 µg/m³ в любой час. Час максимума меняется по месяцам (с 9 до 20 часов), поэтому «один пик в сутки» для всех сезонов неверен. В каждой клетке от 107 до 159 часов, то есть карта опирается на достаточно большое число наблюдений. Это описание, проверка H3 — в задаче 5.1.

![График 3](../figures/03_hour_month_heatmap.png)

## Шаг 9. График 4 — суточный цикл в отопительный сезон и вне его

Вопрос: одинаков ли суточный цикл в отопительный сезон и вне его (H3). Для профиля по часам подходит линия по каждой группе. Вокруг линии рисуем 95 % доверительный интервал (лекция 8): `среднее ± t · std / √n`, то есть диапазон, где с вероятностью 95 % лежит истинное среднее. Оговорка: соседние часы зависимы друг от друга, поэтому настоящий интервал шире нарисованного.

In [13]:
profile = eda.mean_with_ci(hourly, "pm25", ["heating_season", "hour"])
print("минимальное n в группе-часе:", profile["n"].min())

heating = profile[profile["heating_season"]].reset_index(drop=True)
other = profile[~profile["heating_season"]].reset_index(drop=True)
for name, group in [("отопительный сезон", heating), ("вне сезона", other)]:
    peak_row = group.loc[group["mean"].idxmax()]
    low_row = group.loc[group["mean"].idxmin()]
    print(f"{name}: пик {int(peak_row['hour'])}:00 — {peak_row['mean']:.1f}, минимум {int(low_row['hour'])}:00 — {low_row['mean']:.1f} µg/m³")
print("в сезон выше в каждый час:", (heating["mean"].values > other["mean"].values).all())

peak_hour_heating = int(heating.loc[heating["mean"].idxmax(), "hour"])
peak_heating = heating["mean"].max()
peak_other = other["mean"].max()

fig, ax = plt.subplots(figsize=(14, 7))
for flag, group in [(True, heating), (False, other)]:
    ax.plot(group["hour"], group["mean"], marker="o", label=HEATING_LABELS[flag])
    ax.fill_between(group["hour"], group["ci_low"], group["ci_high"], alpha=0.25)
ax.set_xticks(range(0, 24, 3))
ax.set_xlabel("Час (местное время Алматы)")
ax.set_ylabel("Средний PM2.5, µg/m³")
ax.legend()
ax.set_title(f"В отопительный сезон пик PM2.5 в {peak_hour_heating}:00 — {peak_heating:.0f} µg/m³, вне сезона максимум {peak_other:.0f}")
fig.savefig(f"{FIGURES_DIR}/04_hourly_profile.png", dpi=150, bbox_inches="tight")
plt.close(fig)


минимальное n в группе-часе: 797
отопительный сезон: пик 14:00 — 61.1, минимум 6:00 — 27.7 µg/m³
вне сезона: пик 20:00 — 15.9, минимум 4:00 — 9.1 µg/m³
в сезон выше в каждый час: True


**Что видим.** (вывод шага 9) В отопительный сезон пик PM2.5 в 14:00 — 61.1 µg/m³, минимум в 6:00 — 27.7 µg/m³; вне сезона пик в 20:00 — 15.9, минимум в 4:00 — 9.1 µg/m³. В сезон среднее выше в каждый из 24 часов, и интервалы двух групп не пересекаются ни в один час. В каждой точке не меньше 797 часов. Оговорка: соседние часы зависимы, поэтому настоящие интервалы шире нарисованных; строгий ответ даст тест в задаче 5.1.

![График 4](../figures/04_hourly_profile.png)

## Шаг 10. График 5 — связь PM2.5 с ветром и температурой в отопительный сезон

Вопрос: как PM2.5 связан с ветром и температурой (H2). Берём только сутки отопительного сезона: за весь год температура почти повторяет сезон (холодно — значит отопление), и связь «холоднее — грязнее» оказалась бы просто связью «зима — грязнее» (это H1). Внутри сезона видно, добавляют ли что-то ветер и мороз. Подходит диаграмма рассеяния (две числовые величины). Линия тренда — медиана PM2.5 в 10 интервалах по оси X (`pd.qcut` ⚠️: интервалы с примерно равным числом дней), а не линия регрессии: регрессия на скошенных данных тянется за пиками. Связь меряем ρ Спирмена (лекция 9).

In [14]:
heating_days = days[days["heating_season"]]
print("суток отопительного сезона со средним:", len(heating_days))

wind_trend = eda.binned_medians(heating_days, "wind_speed", "pm25_mean")
temp_trend = eda.binned_medians(heating_days, "temp_c", "pm25_mean")
print("Ветер, медианы по интервалам:")
print(wind_trend)
print()
print("Температура, медианы по интервалам:")
print(temp_trend)

season_corr = eda.spearman_matrix(heating_days, ["pm25_mean", "wind_speed", "temp_c"])
rho_wind = season_corr.loc["pm25_mean", "wind_speed"]
rho_temp = season_corr.loc["pm25_mean", "temp_c"]
print()
print(f"ρ Спирмена с pm25_mean: ветер {rho_wind:.2f}, температура {rho_temp:.2f}")

fig, (ax_wind, ax_temp) = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
panels = [
    (ax_wind, "wind_speed", wind_trend, f"Ветер: ρ = {rho_wind:.2f}", "Скорость ветра за сутки, м/с"),
    (ax_temp, "temp_c", temp_trend, f"Температура: ρ = {rho_temp:.2f}", "Температура за сутки, °C"),
]
for ax, column, trend, panel_title, x_label in panels:
    ax.scatter(heating_days[column], heating_days["pm25_mean"], alpha=0.4, s=15)
    ax.plot(trend["x_median"], trend["y_median"], marker="o", linewidth=2.5,
            color=sns.color_palette()[1], label="Медиана по интервалам")
    ax.axhline(WHO_LIMIT, color="black", linestyle="--", label=f"Норма ВОЗ {WHO_LIMIT} µg/m³")
    ax.set_title(panel_title)
    ax.set_xlabel(x_label)
ax_wind.set_ylabel("PM2.5 за сутки, µg/m³")
ax_wind.legend()
fig.suptitle(
    f"В отопительный сезон PM2.5 заметно связан с температурой (ρ = {rho_temp:.2f}), "
    f"с ветром связь слабая (ρ = {rho_wind:.2f})"
)
fig.savefig(f"{FIGURES_DIR}/05_weather_scatter.png", dpi=150, bbox_inches="tight")
plt.close(fig)


суток отопительного сезона со средним: 771
Ветер, медианы по интервалам:
   x_median   y_median   n
0  0.649167  43.208333  78
1  0.970417  47.791667  77
2  1.122083  29.583333  77
3  1.282083  29.416667  77
4  1.401667  25.729167  77
5  1.510417  26.875000  77
6  1.644583  29.687500  77
7  1.808750  27.833333  77
8  2.056667  34.166667  78
9  2.449167  40.562500  76

Температура, медианы по интервалам:
    x_median   y_median   n
0  -9.450000  77.447917  78
1  -5.962500  72.083333  77
2  -3.083333  54.229167  77
3  -1.291667  46.979167  77
4   1.195833  26.875000  77
5   3.304167  33.375000  77
6   5.366667  28.000000  77
7   7.558333  24.000000  77
8  10.420833  21.583333  77
9  14.133333  20.791667  77

ρ Спирмена с pm25_mean: ветер -0.06, температура -0.55


**Что видим.** (вывод шага 10) В отопительный сезон (771 сутки со средним) PM2.5 тем выше, чем холоднее: ρ Спирмена с температурой −0.55, медиана падает с 77.4 µg/m³ в самом холодном интервале (около −9.5 °C) до 20.8 µg/m³ в самом тёплом (около 14.1 °C). С ветром связь слабая: ρ = −0.06, и она не монотонна — медиана 43.2 µg/m³ при самом слабом ветре (около 0.65 м/с), минимум 25.7 при около 1.40 м/с и снова 40.6 при самом сильном (около 2.45 м/с). Это связь, а не причина; влияние ветра и температуры на PM2.5 проверим в 5.1.

![График 5](../figures/05_weather_scatter.png)

## Шаг 11. График 6 — какие факторы сильнее всего связаны с PM2.5

Вопрос: какая погодная величина сильнее связана с PM2.5. Для матрицы связей подходит тепловая карта с числами. Берём ρ Спирмена, а не Пирсона: PM2.5 скошен вправо, связь может быть не линейной, а ранги устойчивы к пикам смога. Все пары считаются на одних и тех же сутках (строки с пропуском в любой колонке убираются). Корреляция не доказывает причину (лекция 9): это разведка, проверка будет в задаче 5.1.

In [15]:
FACTOR_COLUMNS = ["pm25_mean", "temp_c", "wind_speed", "humidity", "precip", "pressure"]
FACTOR_LABELS = {
    "pm25_mean": "PM2.5",
    "temp_c": "Температура",
    "wind_speed": "Ветер",
    "humidity": "Влажность",
    "precip": "Осадки",
    "pressure": "Давление",
}

corr = eda.spearman_matrix(daily, FACTOR_COLUMNS)
print("суток в расчёте:", daily[FACTOR_COLUMNS].dropna().shape[0])

# Столбец PM2.5 за весь год и только за отопительный сезон — рядом
pm25_year = corr["pm25_mean"].drop("pm25_mean")
pm25_season = eda.spearman_matrix(heating_days, FACTOR_COLUMNS)["pm25_mean"].drop("pm25_mean")
compare = pd.DataFrame({"весь год": pm25_year, "отопительный сезон": pm25_season})
print(compare.reindex(pm25_year.abs().sort_values(ascending=False).index).round(2))

# ρ с PM2.5 предыдущих суток: для 6.1 и про зависимость соседних дней
lag_corr = eda.spearman_matrix(daily, ["pm25_mean", "pm25_lag1"])
print("ρ pm25_mean с pm25_lag1:", round(lag_corr.loc["pm25_mean", "pm25_lag1"], 2))

strongest = pm25_year.abs().idxmax()
strongest_rho = pm25_year[strongest]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr.rename(index=FACTOR_LABELS, columns=FACTOR_LABELS),
    annot=True, fmt=".2f", vmin=-1, vmax=1, center=0, cmap="vlag",
    square=True, cbar_kws={"label": "ρ Спирмена"}, ax=ax,
)
ax.set_title(f"Сильнее всего с PM2.5 связана {FACTOR_LABELS[strongest].lower()}: ρ = {strongest_rho:.2f}")
fig.savefig(f"{FIGURES_DIR}/06_corr.png", dpi=150, bbox_inches="tight")
plt.close(fig)


суток в расчёте: 1534
            весь год  отопительный сезон
temp_c         -0.72               -0.55
pressure        0.59                0.28
humidity        0.34               -0.09
precip         -0.17               -0.47
wind_speed     -0.13               -0.06
ρ pm25_mean с pm25_lag1: 0.8


**Что видим.** (вывод шага 11) Расчёт на 1 534 сутках. За весь год сильнее всего с PM2.5 связана температура: ρ = −0.72, затем давление (0.59), влажность (0.34), осадки (−0.17) и ветер (−0.13). Только по отопительному сезону картина другая: температура −0.55, осадки −0.47, давление 0.28, влажность −0.09, ветер −0.06; то есть часть годовых связей — это сезон, а не погода сама по себе, а знак у влажности меняется. ρ с PM2.5 предыдущих суток равна 0.80: соседние сутки сильно зависимы, это важно для 6.1 и для независимости наблюдений в тестах. Корреляция не доказывает причину.

![График 6](../figures/06_corr.png)

## Итог

EDA по графикам 0–7 (числа — в выводе указанных шагов):
- График 0 (шаг 4): суточный PM2.5 скошен вправо (среднее 29.1 больше медианы 16.9 µg/m³, skew 2.48), кривая ECDF отопительного сезона правее.
- График 1 (шаг 5): каждую зиму 30-дневное среднее выше нормы ВОЗ, максимум 135.3 µg/m³ в конце декабря 2020; летом около нормы.
- График 2 (шаг 6): медиана выше нормы с октября по март, хуже всего февраль (58.2 µg/m³); с мая по август ниже нормы.
- График 3 (шаг 8): зимой суточный цикл выражен (максимум 95.3 µg/m³ в январе в 15:00), летом почти плоский.
- График 4 (шаг 9): в отопительный сезон среднее выше в каждый из 24 часов (пик 14:00 — 61.1, минимум 6:00 — 27.7 µg/m³), вне сезона пик 20:00 — 15.9.
- График 5 (шаг 10): в отопительный сезон связь с температурой заметная (ρ = −0.55), с ветром слабая и не монотонная (ρ = −0.06).
- График 6 (шаг 11): за весь год сильнее всего температура (ρ = −0.72); по отопительному сезону связи слабее, ρ с предыдущими сутками 0.80.
- График 7 (шаг 7): норма ВОЗ превышена в 845 сутках из 1 534 (55.1 %), в декабре в 96 % дней, в августе в 4 %.

**Что проверим тестами в 5.1.**
- H1 (сезон) и H2 (ветер, температура): тест Welch и Mann–Whitney (распределение скошено, шаг 4), χ² «сезон × превышение нормы».
- H3: суточный цикл виден на графиках 3–4.

**Открытый вопрос.** Мы не знаем, почему профиль PM2.5 по часам имеет пик около 14:00 и минимум около 6:00. Метки времени погоды по местному времени проверены (минимум температуры в 6–7 часов, максимум в 14–15). Причину формы профиля пока не выясняли.

**Ограничения.**
- Без `pm25` 20.86 % часов и 510 суток; дыра лета 2022 (май–август), поэтому месячные статистики объединяют разное число лет.
- Интервалы на графике 4 узкие, потому что соседние часы зависимы; настоящие шире.
- Корреляции за весь год смешаны с сезоном (график 6: связи по отопительному сезону слабее).
- Одна станция.

Раздел тестов (5.1) будет добавлен ниже.